In [ ]:
import sys
from pathlib import Path
import torch
import numpy as np

ROOT = Path.cwd().resolve().parent
if str(ROOT / "src") not in sys.path:
    sys.path.append(str(ROOT / "src"))

from project_paths import get_paths
from teacher_finetune_headtail import (
    TeacherModelConfig, 
    build_teacher_model, 
    build_teacher_tokenizer, 
    get_llrd_optimizer_parameters,
    WeightedCETrainer,
    compute_metrics,
    calculate_class_weights,
    bf16_supported
)
from transformers import DataCollatorWithPadding, TrainingArguments
from datasets import load_from_disk

paths = get_paths(ROOT)
DATA_DIR = paths.data_processed

print(f"Data Source: {DATA_DIR}")
print(f"Device: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU'}")

c:\Users\cola0\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


KeyboardInterrupt: 

In [ ]:

try:
    train_ds = load_from_disk(str(DATA_DIR / "train.hf"))
    val_ds = load_from_disk(str(DATA_DIR / "val.hf"))
except:
    from datasets import load_dataset
    print("Loading from Parquet files...")
    train_ds = load_dataset("parquet", data_files=str(DATA_DIR / "train.parquet"))["train"]
    val_ds = load_dataset("parquet", data_files=str(DATA_DIR / "val.parquet"))["train"]

print(f"Train size: {len(train_ds)}")
print(f"Val size:   {len(val_ds)}")

In [ ]:
class_weights = calculate_class_weights(train_ds)

print(f"--- Class Weight Configuration ---")
print(f"Class weights [w0,w1]: {class_weights.tolist()}")
if 0.9 < class_weights[1] < 1.1:
    print(">> Dataset appears Balanced. Standard training applies.")
else:
    print(">> Dataset appears Imbalanced. Weighted Loss will be used.")

In [ ]:
MODEL_NAME = "bert-base-uncased"

tokenizer = build_teacher_tokenizer(MODEL_NAME)
config = TeacherModelConfig(
    model_name=MODEL_NAME,
    gradient_checkpointing=True 
)
model = build_teacher_model(config)

LR = 2e-5
WEIGHT_DECAY = 0.01
optimizer_grouped_parameters = get_llrd_optimizer_parameters(
    model, learning_rate=LR, weight_decay=WEIGHT_DECAY, layer_decay=0.95
)
optimizer = torch.optim.AdamW(optimizer_grouped_parameters)

In [ ]:
BATCH_SIZE = 8        
ACCUMULATION = 3      
EPOCHS = 3

training_args = TrainingArguments(
    output_dir=str(paths.checkpoints / "bert_teacher_finetuned"),
    
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=BATCH_SIZE * 2, 
    gradient_accumulation_steps=ACCUMULATION,
    num_train_epochs=EPOCHS,
    
    fp16=True, 

    logging_steps=100,
    eval_strategy="steps", 
    eval_steps=2000,
    save_strategy="steps", 
    save_steps=2000,
    save_total_limit=4,
    load_best_model_at_end=True,
    metric_for_best_model="pr_auc", 
    greater_is_better=True,
    report_to="none",
    warmup_ratio=0.06,
    dataloader_num_workers=0,
    group_by_length=True 
)

collator = DataCollatorWithPadding(tokenizer)

In [ ]:
from transformers import get_linear_schedule_with_warmup

num_training_steps = (len(train_ds) // (BATCH_SIZE * ACCUMULATION)) * EPOCHS
num_warmup_steps = int(0.06 * num_training_steps)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=num_warmup_steps,
    num_training_steps=num_training_steps
)


trainer = WeightedCETrainer(
    class_weights=class_weights, 
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    tokenizer=tokenizer,
    data_collator=collator,
    compute_metrics=compute_metrics,
    optimizers=(optimizer, None) 
)

print("Starting Training...")
trainer.train()

In [ ]:
import numpy as np
import torch
from sklearn.metrics import f1_score, precision_score, recall_score

pred_out = trainer.predict(val_ds) 

logits = torch.tensor(pred_out.predictions)
probs = torch.softmax(logits, dim=-1).numpy()[:, 1] 

labels = pred_out.label_ids.astype(int)

thresholds = np.linspace(0.01, 0.99, 99)
best = {"threshold": 0.5, "f1": -1, "precision": None, "recall": None}

for t in thresholds:
    preds = (probs >= t).astype(int)
    f1 = f1_score(labels, preds, pos_label=1, zero_division=0)
    
    if f1 > best["f1"]:
        best["threshold"] = float(t)
        best["f1"] = float(f1)
        best["precision"] = float(precision_score(labels, preds, pos_label=1, zero_division=0))
        best["recall"] = float(recall_score(labels, preds, pos_label=1, zero_division=0))

print("Best threshold on validation:", best)

In [ ]:
import numpy as np
import torch
from datasets import load_dataset
from sklearn.metrics import f1_score, precision_score, recall_score, roc_auc_score, average_precision_score

t = best["threshold"] 

test_ds = load_dataset("parquet", data_files=str(DATA_DIR / "test.parquet"))["train"]

test_out = trainer.predict(test_ds)

logits = torch.tensor(test_out.predictions)
probs = torch.softmax(logits, dim=-1).numpy()[:, 1]

y_true = test_out.label_ids.astype(int)
y_pred = (probs >= t).astype(int)

print(f"--- TEST METRICS (Threshold: {t:.4f}) ---")
print("TEST F1:", f1_score(y_true, y_pred, pos_label=1, zero_division=0))
print("TEST Precision:", precision_score(y_true, y_pred, pos_label=1, zero_division=0))
print("TEST Recall:", recall_score(y_true, y_pred, pos_label=1, zero_division=0))
print("TEST ROC-AUC:", roc_auc_score(y_true, probs))
print("TEST PR-AUC:", average_precision_score(y_true, probs))

In [ ]:
import matplotlib.pyplot as plt

log_history = trainer.state.log_history

train_steps, train_loss = [], []
eval_steps, eval_loss = [], []

for row in log_history:
    if "loss" in row and "eval_loss" not in row:   
        train_steps.append(row.get("step", None))
        train_loss.append(row["loss"])
    if "eval_loss" in row:                         
        eval_steps.append(row.get("step", None))
        eval_loss.append(row["eval_loss"])

plt.figure()
plt.plot(train_steps, train_loss, label="train loss")
plt.plot(eval_steps, eval_loss, label="eval loss")
plt.xlabel("step")
plt.ylabel("loss")
plt.title("Train/Eval loss vs step")
plt.legend()
plt.show()


In [ ]:
import numpy as np

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

pred_out = trainer.predict(val_ds)

logits = pred_out.predictions.reshape(-1)
labels = pred_out.label_ids.astype(int)
probs  = sigmoid(logits)


from sklearn.metrics import precision_recall_curve, average_precision_score

precision, recall, pr_thresholds = precision_recall_curve(labels, probs, pos_label=1)
pr_auc = average_precision_score(labels, probs)

plt.figure()
plt.plot(recall, precision, label=f"PR curve (AP={pr_auc:.4f})")
plt.xlabel("Recall")
plt.ylabel("Precision")
plt.title("Precision–Recall curve")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

fpr, tpr, roc_thresholds = roc_curve(labels, probs, pos_label=1)
roc_auc = roc_auc_score(labels, probs)

plt.figure()
plt.plot(fpr, tpr, label=f"ROC curve (AUC={roc_auc:.4f})")
plt.plot([0, 1], [0, 1], linestyle="--", label="random")
plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC curve")
plt.legend()
plt.show()


In [ ]:
from sklearn.metrics import f1_score, precision_score, recall_score

thresholds = np.linspace(0.01, 0.99, 99)
P, R, F = [], [], []

for t in thresholds:
    pred = (probs >= t).astype(int)
    P.append(precision_score(labels, pred, pos_label=1, zero_division=0))
    R.append(recall_score(labels, pred, pos_label=1, zero_division=0))
    F.append(f1_score(labels, pred, pos_label=1))

best_t = thresholds[int(np.argmax(F))]

plt.figure()
plt.plot(thresholds, P, label="Precision")
plt.plot(thresholds, R, label="Recall")
plt.plot(thresholds, F, label="F1")
plt.axvline(0.58, linestyle="--", label="chosen threshold 0.58")
plt.axvline(best_t, linestyle=":", label=f"best F1 threshold {best_t:.2f}")
plt.xlabel("Threshold")
plt.ylabel("Score")
plt.title("Precision/Recall/F1 vs threshold (positive=spoiler)")
plt.legend()
plt.show()

print("Best threshold by scan:", best_t, "Best F1:", max(F))
